In [ ]:
import pandas as pd
import itertools
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from tmtools.io import get_structure, get_residue_data
from tmtools import tm_align

seq_file = 'grampa_s_aureus_7_25_with_GRAMPA.csv'   
pdb_dir = './pdb'                                   
output_file = 'AMPCliff_pairs_TMtools_0_9.csv'      
TMSCORE_THRESHOLD = 0.9 
value_DIFF_THRESHOLD = 1

df = pd.read_csv(seq_file)
all_rows = list(df.itertuples(index=False))

from tmtools.io import get_structure, get_residue_data
from tmtools import tm_align
def calc_tm_score(pdb1, pdb2):
    structure1 = get_structure(pdb1)
    structure2 = get_structure(pdb2)
    chain1 = next(structure1.get_chains())
    chain2 = next(structure2.get_chains())
    coords1, seq1 = get_residue_data(chain1)
    coords2, seq2 = get_residue_data(chain2)
    # Calculate the TM-score using tmtools
    result = tm_align(coords1, coords2, seq1, seq2)
    return result.tm_norm_chain1

pair_iter = itertools.combinations(all_rows, 2)

pairs = []
for row1, row2 in tqdm(pair_iter, total=100):
    id1, id2 = row1.ID, row2.ID
    pdb1 = f"{pdb_dir}/{id1}.pdb"
    pdb2 = f"{pdb_dir}/{id2}.pdb"
    try:
        tm_score = calc_tm_score(pdb1, pdb2)
    except Exception as e:
        print(f"TM-score error for {id1} vs {id2}: {e}")
        continue
    value_diff = abs(row1.value - row2.value)
    pairs.append({
        'ID1': id1,
        'ID2': id2,
        'TMscore': tm_score,
        'value1': row1.value,
        'value2': row2.value,
        'valueDiff': value_diff
    })

pairs_df = pd.DataFrame(pairs)

cliff_pairs = pairs_df[
    (pairs_df['TMscore'] >= TMSCORE_THRESHOLD) &
    (pairs_df['valueDiff'] >= np.log10(value_DIFF_THRESHOLD))
]
cliff_pairs.to_csv(output_file, index=False)

id2len = dict(zip(df['ID'].astype(str), df['Sequence'].str.len()))

cliff_pairs = cliff_pairs.copy()

cliff_pairs['ID1'] = cliff_pairs['ID1'].astype(str)
cliff_pairs['ID2'] = cliff_pairs['ID2'].astype(str)

cliff_pairs['Len1'] = cliff_pairs['ID1'].map(id2len)
cliff_pairs['Len2'] = cliff_pairs['ID2'].map(id2len)
cliff_pairs['MinLength'] = cliff_pairs[['Len1', 'Len2']].min(axis=1)

def length_bin(length):
    if 7 <= length <= 9:
        return '7-9'
    elif 10 <= length <= 12:
        return '10-12'
    elif 13 <= length <= 15:
        return '13-15'
    elif 16 <= length <= 18:
        return '16-18'
    elif 19 <= length <= 21:
        return '19-21'
    elif 22 <= length <= 25:
        return '22-25'
    else:
        return 'other'

cliff_pairs['LengthBin'] = cliff_pairs['MinLength'].apply(length_bin)

bin_counts = cliff_pairs['LengthBin'].value_counts().sort_index()
print("AMP Cliff pairs：")
print(bin_counts)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 7))
plt.pie(
    bin_counts, 
    labels=bin_counts.index, 
    autopct=lambda pct: int(round(pct * bin_counts.sum() / 100)), 
    startangle=90
)
plt.title('Distribution of Structure-based AMP Cliff Pairs by Peptide Length (Min)')
plt.axis('equal')
plt.show()

In [ ]:
import pandas as pd

TMscore_threshold = 0.93

def format_score(score):
    return str(score).replace('.', '_')  # 0.93 -> '0_93'

score_str = format_score(TMscore_threshold)

input_file = f"./AMPCliff_pairs_TMtools_0_9.csv" 
output_file = f"./AMPCliff_pairs_TMtools_{score_str}.csv"

df = pd.read_csv(input_file)
filtered_df = df[df['TMscore'] >= TMscore_threshold]

filtered_df.to_csv(output_file, index=False)
print(f"筛选后的数据已保存到: {output_file}")

In [ ]:
import pandas as pd
import math
import os

TMscore_threshold = 0.93             # TM-score 0.93
threshold_value = 50                 # activity 5 10 20 30 40 50

def format_score(score):
    return str(score).replace('.', '_')  # 0.93 -> '0_9_3'

score_str = format_score(TMscore_threshold)

file_path = f'./AMPCliff_pairs_TMtools_{score_str}.csv'

df = pd.read_csv(file_path)

if 'valueDiff' not in df.columns or 'TMscore' not in df.columns:
    print("error")
else:
    log_threshold = math.log10(threshold_value)
    print(f'log10({threshold_value}) = {log_threshold}')
    filtered_df = df[(df['valueDiff'] > log_threshold) & (df['TMscore'] >= TMscore_threshold)]
    print(f'{filtered_df.shape[0]}')

    output_folder = f'./{score_str}__{threshold_value}_structure/'
    os.makedirs(output_folder, exist_ok=True)

    output_file_path = os.path.join(output_folder, 'AMPCliff_pairs_TMtools.csv')

    filtered_df.to_csv(output_file_path, index=False)
    print(filtered_df.head())

In [ ]:
import pandas as pd
import math
import os

TMscore_threshold = 0.93
threshold_value = 50  #5 10 20 30 40 50

def format_score(score):
    return str(score).replace('.', '_')  # 0.93 -> 0_93

score_str = format_score(TMscore_threshold)

pair_file_path = f'./{score_str}__{threshold_value}_structure/AMPCliff_pairs_TMtools.csv'
full_peptides_path = './grampa_s_aureus_7_25_with_GRAMPA.csv'

output_folder = f'./{score_str}__{threshold_value}_structure/'
output_test_file = os.path.join(output_folder, 'test.csv')
output_train_file = os.path.join(output_folder, 'train.csv')

df = pd.read_csv(pair_file_path)

if 'valueDiff' not in df.columns:
    print("error")
else:
    log_threshold = math.log10(threshold_value)
    print(f'log10({threshold_value}) = {log_threshold}')
    
    filtered_df = df[df['valueDiff'] > log_threshold]
    print(f'{filtered_df.shape[0]}')

    filtered_df.to_csv(pair_file_path, index=False)
    print(f"{pair_file_path}")

    df1 = filtered_df[['ID1', 'value1']].rename(columns={'ID1': 'ID', 'value1': 'value'})
    df2 = filtered_df[['ID2', 'value2']].rename(columns={'ID2': 'ID', 'value2': 'value'})
    test_df = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset='ID')
    test_df.to_csv(output_test_file, index=False)
    print(f"test.csv saved to：{output_test_file}")
    full_df = pd.read_csv(full_peptides_path)
    if 'ID' not in full_df.columns or 'value' not in full_df.columns:
        raise ValueError("error: 'ID' and 'value' 列")

    train_df = full_df[~full_df['ID'].isin(test_df['ID'])][['ID', 'value']]
    train_df.to_csv(output_train_file, index=False)
    print(f"train.csv saved to：{output_train_file}")
